# NB06e — Hierarchical Clustering and Gaussian Mixtures

**Role:** Compact-extension  
**Manual section:** 2.3.2.b — Hierarchical clustering and Gaussian mixtures  
**Prerequisites:** NB06d (K-Means and PCA)

---

## Purpose of this notebook

This compact-extension notebook expands the unsupervised toolkit beyond K-Means. You will see two methods that address limitations of K-Means: hierarchical clustering reveals a nested structure of similarity, and Gaussian mixtures provide soft (probabilistic) assignments instead of hard cluster labels.

**Dataset:** `dataset_D_unsupervised_extension.csv` (sampled to 800 rows)

**Finance motivation:** In wealth management and customer segmentation, clients rarely fit neatly into one category. A customer may be 70% "conservative saver" and 30% "growth seeker." Gaussian mixtures capture this ambiguity formally. Similarly, hierarchical clustering can reveal that two broad segments each contain meaningful sub-segments — information that a flat K-Means partition would miss.

## Section 0 — Why go beyond K-Means

K-Means is an effective starting point for segmentation (see NB06d), but it has two important limitations:

1. **Hard assignment only.** Every observation is assigned to exactly one cluster. In finance, many entities (customers, assets, transactions) sit between categories.
2. **Flat structure.** K-Means produces k groups with no hierarchy. It cannot tell you whether two groups are very similar to each other but different from a third.

Hierarchical clustering and Gaussian mixtures each address one of these limitations:

| Method | What it adds | Typical finance use |
|--------|-------------|-------------------|
| **Hierarchical clustering** | Nested similarity structure (dendrogram) | Sub-segmentation, organisational groupings |
| **Gaussian mixtures** | Soft probabilistic assignment | Ambiguous customer profiles, risk grading |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

from pathlib import Path
_candidates = [Path('data/processed'), Path('../data/processed')]
DATA_DIR = next((p for p in _candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Cannot locate data/processed/. Launch from project root or notebooks/.')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from scipy.cluster.hierarchy import linkage, dendrogram


In [ ]:
df = pd.read_csv(DATA_DIR / 'dataset_D_unsupervised_extension.csv').sample(n=800, random_state=42).copy()
scaler = StandardScaler()
X = scaler.fit_transform(df)
df.head()

## Section 1 — Hierarchical Clustering

### How agglomerative clustering works

Agglomerative clustering starts with **each observation as its own cluster** and then iteratively merges the two closest clusters until only one remains. The key decision is the **linkage method**, which defines how "closeness" between clusters is measured:

| Linkage | Distance between clusters A and B | Behaviour |
|---------|----------------------------------|-----------|
| **Single** | Minimum distance between any pair of points | Tends to produce elongated chains |
| **Complete** | Maximum distance between any pair of points | Produces compact, similar-sized clusters |
| **Average** | Average distance between all pairs of points | Balanced compromise |
| **Ward** | Increase in total within-cluster variance after merging | Tends to produce similar-sized, spherical clusters |

In this notebook we use **Ward linkage**, which is a sensible default for many segmentation tasks.

In [ ]:
subset = X[:60]
Z = linkage(subset, method='ward')
plt.figure(figsize=(9,4))
dendrogram(Z, no_labels=True, color_threshold=None)
plt.title('Dendrogram on a compact sample')
plt.show()

### How to read a dendrogram

A dendrogram visualises the entire merging history of the clustering process:

- **Each leaf** at the bottom represents one observation (or a small group).
- **Each horizontal line** represents a merge event. The **height** (y-axis) at which two branches join indicates the **distance** at which they were merged — higher means less similar.
- **Cutting the dendrogram at a chosen height** defines the number of clusters. A horizontal cut at a lower level produces more clusters; at a higher level, fewer.

**Practical guidance:** look for large vertical gaps in the dendrogram. These suggest natural cluster boundaries — the algorithm had to "reach far" to merge those groups, meaning they are genuinely different.

In [ ]:
agg = AgglomerativeClustering(n_clusters=3)
labels_h = agg.fit_predict(X)
df_h = df.copy()
df_h['cluster_h'] = labels_h
df_h['cluster_h'].value_counts().sort_index()

## Section 2 — Gaussian Mixtures

### How Gaussian Mixture Models work

Unlike K-Means, which assigns each observation to exactly one cluster (hard assignment), a Gaussian Mixture Model (GMM) assumes that the data was generated by a **mixture of Gaussian distributions** — one per cluster.

Each cluster is characterised by:
- a **mean** (centre),
- a **covariance matrix** (shape and orientation),
- a **mixing weight** (how large that cluster is relative to the others).

For each observation, the model computes the **probability of belonging to each cluster**. This is called **soft assignment**: a customer might have a 72% probability of belonging to segment A and 28% to segment B, rather than being forced into one group.

**Why this matters in finance:** soft assignment is more realistic for customer segmentation, risk classification and portfolio grouping, where entities often share characteristics of multiple categories.

In [ ]:
gmm = GaussianMixture(n_components=3, random_state=42)
gmm.fit(X)
labels_g = gmm.predict(X)
probs_g = gmm.predict_proba(X)
df_g = df.copy()
df_g['cluster_gmm'] = labels_g
df_g['max_assignment_probability'] = probs_g.max(axis=1)
df_g[['cluster_gmm', 'max_assignment_probability']].head()

### Interpreting soft assignments

The `max_assignment_probability` column shows how confident the model is about each customer's cluster membership:

- **High probability (> 0.90):** the customer fits clearly into one segment. Their profile is typical of that group.
- **Moderate probability (0.60–0.90):** the customer is primarily in one segment but shares some characteristics with another. These "boundary" customers deserve attention in a business context.
- **Low maximum probability (< 0.60):** the customer is genuinely ambiguous — they do not fit neatly into any single segment.

In a financial advisory context, ambiguous customers may benefit from a more personalised approach rather than a standard segment-based treatment. This is one of the main advantages of soft assignment over hard clustering.

In [ ]:
df_g['cluster_gmm'].value_counts().sort_index()

## Section 3 — Compare segment profiles

The goal of clustering is not just to create labels. The goal is to learn whether the clusters make business sense.

In [ ]:
profiles_h = df_h.groupby('cluster_h').mean(numeric_only=True).round(2)
profiles_h

In [ ]:
profiles_g = df_g.groupby('cluster_gmm').mean(numeric_only=True).round(2)
profiles_g

In [ ]:
comparison_cols = ['balance', 'age', 'estimated_salary', 'digital_usage_proxy', 'complaints_last_6m']
profiles_h[comparison_cols].T.plot(kind='bar')
plt.title('Hierarchical clustering — segment profiles')
plt.xticks(rotation=30)
plt.show()

## What you have learned

- **Hierarchical clustering** reveals nested similarity structures through a dendrogram. Linkage method and cut height are the key decisions.
- **Gaussian mixtures** provide probabilistic (soft) cluster assignments, capturing ambiguity that K-Means cannot express.
- Both methods use the same scaled data as K-Means (NB06d), making comparison straightforward.
- In finance, soft assignment and hierarchical structure often reflect business reality better than hard, flat partitions.

### When to use which method

| If you need… | Consider… |
|-------------|----------|
| A quick, interpretable segmentation | K-Means (NB06d) |
| To understand sub-group relationships | Hierarchical clustering |
| Probabilistic segment membership | Gaussian mixtures |

---

*This is a compact-extension notebook supporting manual section 2.3.2.b.*